Setup Project Path

In [ ]:

import sys
from pathlib import Path
import os

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


 Import Required Modules

In [ ]:

import time
import random
import pandas as pd
import numpy as np
import os

from model.model_implement import run_pipeline_for_experiment


 Read Memory Limit from Environment

In [ ]:
import os

def parse_memory_mb(mem_str):
    if mem_str.endswith('g'):
        return int(mem_str[:-1]) * 1024
    elif mem_str.endswith('m'):
        return int(mem_str[:-1])
    else:
        return int(mem_str)

memory_mb_str = os.environ.get("MEMORY_MB", "512")
memory_mb = parse_memory_mb(memory_mb_str)
print(f"Running experiment with memory limit: {memory_mb} MB")


 Dataset Sampling

In [ ]:

REPORTS_DIR = "../report/reports"

# Collect all PDF files
pdf_files = [os.path.join(root, f)
             for root, _, files in os.walk(REPORTS_DIR)
             for f in files if f.endswith(".pdf")]

print(f"Total PDFs found: {len(pdf_files)}")

# Sample 10 PDFs randomly
random.seed(42)
SELECTED_PDFS = random.sample(pdf_files, min(10, len(pdf_files)))

# Batch PDFs
BATCH_SIZE = 10
PDF_BATCHES = [SELECTED_PDFS[i:i+BATCH_SIZE]
               for i in range(0, len(SELECTED_PDFS), BATCH_SIZE)]


Experiment Runner Function

In [ ]:

def run_memory_capped_experiment(memory_limit_mb, pdf_batch):
    results = []

    for pdf in pdf_batch:
        start_time = time.time()
        status = "success"
        latency = None
        output_length = 0
        partial_output = False
        error_type = None

        try:
            result = run_pipeline_for_experiment(
                pdf_path=pdf,
                language="English",
                model="mistral"
            )
            latency = time.time() - start_time

            if result is None or "error" in result:
                status = "failure"
            else:
                output_length = result.get("output_length", 0)
                partial_output = output_length < 300

        except Exception as e:
            latency = time.time() - start_time
            status = "failure"
            error_type = str(e)

        results.append({
            "pdf_name": os.path.basename(pdf),
            "memory_mb": memory_limit_mb,
            "status": status,
            "latency_sec": latency,
            "output_length": output_length,
            "partial_output": partial_output,
            "error_type": error_type
        })

    return results


Run Experiment

In [ ]:

all_results = []

for batch_id, batch in enumerate(PDF_BATCHES):
    print(f"Running batch {batch_id+1}/{len(PDF_BATCHES)}")
    batch_results = run_memory_capped_experiment(memory_mb, batch)

    # Add batch_id
    for r in batch_results:
        r["batch_id"] = batch_id + 1
        all_results.append(r)

df = pd.DataFrame(all_results)


 Save Results to CSV

In [ ]:
OUTPUT_CSV = f"memory_pressure_results_{memory_mb}MB.csv"
df.to_csv(OUTPUT_CSV, index=False)
print(f"Results saved to {OUTPUT_CSV}")